# EcoHome Energy Advisor - RAG Setup

In this notebook, you'll set up the Retrieval-Augmented Generation (RAG) pipeline for the EcoHome Energy Advisor. This will allow the agent to access and cite relevant energy-saving tips and best practices.

## Learning Objectives
- Set up ChromaDB vector store
- Load and process energy-saving documents
- Create embeddings for document chunks
- Implement semantic search functionality
- Test the RAG pipeline

## Documents Available
The setup automatically indexes every `.txt` article in `data/documents/`,
including device, HVAC, automation, renewable-energy, seasonal, and storage guidance.


## 1. Import Required Libraries


In [1]:
# Import the necessary libraries for RAG setup
import os
from pathlib import Path

from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

from tools import get_embeddings

In [2]:
load_dotenv()

True

## 2. Load and Process Documents


In [3]:
# Load every energy-saving text document in data/documents.
documents = []
document_directory = Path("data/documents")
# api: glob("*.txt") discovers every text article in the directory.
document_paths = sorted(document_directory.glob("*.txt"))

if not document_paths:
    raise ValueError(f"No knowledge documents found in {document_directory}")

for doc_path in document_paths:
    loader = TextLoader(str(doc_path))
    docs = loader.load()
    documents.extend(docs)
    print(f"Loaded {len(docs)} documents from {doc_path}")

print(f"Total documents loaded: {len(documents)}")


Loaded 1 documents from data/documents/tip_device_best_practices.txt
Loaded 1 documents from data/documents/tip_energy_savings.txt
Loaded 1 documents from data/documents/tip_energy_storage_optimization.txt
Loaded 1 documents from data/documents/tip_hvac_optimization.txt
Loaded 1 documents from data/documents/tip_renewable_energy_integration.txt
Loaded 1 documents from data/documents/tip_seasonal_energy_management.txt
Loaded 1 documents from data/documents/tip_smart_home_automation.txt
Total documents loaded: 7


## 3. Split Documents into Chunks


In [4]:
# Split documents into smaller chunks for better retrieval
# Use RecursiveCharacterTextSplitter with appropriate chunk_size and chunk_overlap
# Experiment with different chunk sizes (e.g., 500, 1000, 1500 characters)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Split the documents
splits = text_splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(splits)} chunks")

# Show sample chunk
if splits:
    print("\nSample chunk (first 200 characters):")
    print(splits[0].page_content[:200] + "...")


Split 7 documents into 19 chunks

Sample chunk (first 200 characters):
Large devices like electric vehicles, washing machines and dishwashers often support delayed start or timer functions. Schedule these devices to run outside of peak electricity pricing hours or during...


## 4. Create Vector Store


In [5]:
# Create a ChromaDB vector store
# Initialize OpenAIEmbeddings
# Create the vector store with the document chunks
# Persist the vector store to disk for future use

# Set up the persist directory
persist_directory = "data/vectorstore"
os.makedirs(persist_directory, exist_ok=True)

# Fail loudly rather than persisting an empty store. A wrong document path makes
# the loader above find nothing, and Chroma will happily build a store with zero
# vectors -- which then looks like "retrieval is bad" instead of "no data".
if not splits:
    raise ValueError(
        "No document chunks to embed. Check the paths in `document_paths` above "
        "-- the tip files live in data/documents/ (plural)."
    )

# Initialize embeddings. The shared helper ensures the store is written
# and read with the same endpoint, key, and model.
embeddings = get_embeddings()

# Create the vector store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Vector store created and persisted to {persist_directory}")
print(f"Total vectors stored: {len(splits)}")

Vector store created and persisted to data/vectorstore
Total vectors stored: 19


## 5. Test the RAG Pipeline


In [6]:
# Test the search functionality
# Try different queries related to energy optimization
# Test queries like:
# - "electric vehicle charging tips"
# - "thermostat optimization"
# - "dishwasher energy saving"
# - "solar power maximization"

test_queries = [
    "electric vehicle charging tips",
    "thermostat optimization",
    "dishwasher energy saving",
    "solar power maximization",
    "HVAC system efficiency",
    "pool pump scheduling"
]

print("=== Testing Vector Search ===")
for query in test_queries:
    print(f"\nQuery: '{query}'")
    docs = vectorstore.similarity_search(query, k=2)
    for i, doc in enumerate(docs):
        print(f"  Result {i+1}: {doc.page_content[:100]}...")


=== Testing Vector Search ===

Query: 'electric vehicle charging tips'


  Result 1: Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
  Result 2: Coordinate the battery with flexible devices. Direct solar production into an electric vehicle, wate...

Query: 'thermostat optimization'


  Result 1: Use a schedule that reflects occupancy. Reduce heating or cooling when the home is empty, but begin ...
  Result 2: Use power strips to easily turn off multiple devices at once. Many electronics continue to draw powe...

Query: 'dishwasher energy saving'


  Result 1: Dishwasher Best Practices:
- Only run when completely full
- Use the energy-saving or eco mode when ...
  Result 2: Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...

Query: 'solar power maximization'


  Result 1: Compare the solar opportunity with the electricity tariff. If midday grid prices are low, moving a l...
  Result 2: Integrating Rooftop Solar with Household Energy Use

Rooftop photovoltaic generation has the greates...

Query: 'HVAC system efficiency'


  Result 1: Use a schedule that reflects occupancy. Reduce heating or cooling when the home is empty, but begin ...
  Result 2: Evaluate changes with comparable data. Compare energy use across days with similar outdoor condition...

Query: 'pool pump scheduling'


  Result 1: In cold periods, protect heat-pump efficiency. Use gradual thermostat recovery and avoid changes tha...
  Result 2: Seasonal Energy Management for Smart Homes

An energy schedule should change with daylight, weather ...


## 6. Test the Search Tool


In [7]:
# Test the search_energy_tips tool from tools.py
# Import and test the tool with various queries
# Verify that it returns relevant results

from tools import search_energy_tips

# Test the search_energy_tips function
print("=== Testing search_energy_tips Tool ===")

test_queries = [
    "electric vehicle charging",
    "thermostat settings",
    "dishwasher optimization",
    "solar power tips"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    result = search_energy_tips.invoke(
        input={
            "query": query, 
            "max_results": 3,
        }
    )
    
    if "error" in result:
        print(f"  Error: {result['error']}")
    else:
        print(f"  Found {result['total_results']} results")
        for i, tip in enumerate(result['tips']):
            print(f"    {i+1}. {tip['content'][:100]}...")
            print(f"       Source: {tip['source']}")
            print(f"       Relevance: {tip['relevance_score']}")


=== Testing search_energy_tips Tool ===

Query: 'electric vehicle charging'


  Found 3 results
    1. Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
       Source: data/documents/tip_device_best_practices.txt
       Relevance: high
    2. Coordinate the battery with flexible devices. Direct solar production into an electric vehicle, wate...
       Source: data/documents/tip_energy_storage_optimization.txt
       Relevance: high
    3. Account for losses and battery wear. Charging one kilowatt-hour does not normally make the full kilo...
       Source: data/documents/tip_energy_storage_optimization.txt
       Relevance: medium

Query: 'thermostat settings'


  Found 3 results
    1. Use a schedule that reflects occupancy. Reduce heating or cooling when the home is empty, but begin ...
       Source: data/documents/tip_hvac_optimization.txt
       Relevance: high
    2. Use power strips to easily turn off multiple devices at once. Many electronics continue to draw powe...
       Source: data/documents/tip_energy_savings.txt
       Relevance: high
    3. In cold periods, protect heat-pump efficiency. Use gradual thermostat recovery and avoid changes tha...
       Source: data/documents/tip_seasonal_energy_management.txt
       Relevance: medium

Query: 'dishwasher optimization'


  Found 3 results
    1. Dishwasher Best Practices:
- Only run when completely full
- Use the energy-saving or eco mode when ...
       Source: data/documents/tip_device_best_practices.txt
       Relevance: high
    2. Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
       Source: data/documents/tip_device_best_practices.txt
       Relevance: high
    3. Prevent simultaneous starts from creating a new household peak. A simple priority order can stagger ...
       Source: data/documents/tip_smart_home_automation.txt
       Relevance: medium

Query: 'solar power tips'


  Found 3 results
    1. In cold periods, protect heat-pump efficiency. Use gradual thermostat recovery and avoid changes tha...
       Source: data/documents/tip_seasonal_energy_management.txt
       Relevance: high
    2. Coordinate the battery with flexible devices. Direct solar production into an electric vehicle, wate...
       Source: data/documents/tip_energy_storage_optimization.txt
       Relevance: high
    3. Compare the solar opportunity with the electricity tariff. If midday grid prices are low, moving a l...
       Source: data/documents/tip_renewable_energy_integration.txt
       Relevance: medium
